In [ ]:
import os
import subprocess


import matplotlib.pyplot as plt
import seaborn as sns
from tueplots import bundles
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib as mpl

print(os.environ['TEXMFHOME'])

# plt.rcParams.update(bundles.icml2022())
# set dpi to 300
plt.rcParams['savefig.dpi'] = 300

SAVE_FLAG = True

def save_fig(name: str):
    # save figure to figures - pdf and png
    if SAVE_FLAG:
        os.makedirs(f'figures/{name}', exist_ok=True)
        plt.savefig(f'figures/{name}/{name}.pdf')
        plt.savefig(f'figures/{name}/{name}.png')
    else:
        plt.show()

    plt.clf()

os.makedirs('figures', exist_ok=True)

def is_significant(lb, ub):
    if lb < 1 and ub < 1 or lb > 1 and ub > 1:
        return True
    return False


# Misinfo. discernment

We look at responses to the control group and understand the participants discernment of misinformation.

Below is all participants

In [ ]:
misinfo_discern_control_file_path = "results/control_group_misinfo_discernment_with_metrics_unsure.csv"

misinfo_discern_control_df = pd.read_csv(misinfo_discern_control_file_path)

misinfo_discern_metrics_names = misinfo_discern_control_df['observed']
misinfo_discern_odds = misinfo_discern_control_df['OR'].apply(lambda x: float(x))
misinfo_discern_lb = misinfo_discern_control_df['LB'].apply(lambda x: float(x))
misinfo_discern_ub = misinfo_discern_control_df['UB'].apply(lambda x: float(x))

metric_map = {
    'Correct': 'Strict \n Accuracy',
    'Correct+Unsure': 'Lenient \n Accuracy',
    'Trust': 'Trust',
    'Adherence': 'Strict \n Adherence',
    'Adherence+Unsure': 'Lenient \n Adherence',
    'Unsure': 'Unsure',
}

colors = ['C1','C2','C4','C5','C6','C7']

# Metrics should be presented in the order - Correct, Trust, Adherence, Unsure. Re-order names, odds, lb, ub
indices_remap = [misinfo_discern_metrics_names.tolist().index(metric) for metric in metric_map]
misinfo_discern_metrics_names = [metric_map[misinfo_discern_metrics_names[i]] for i in indices_remap]
misinfo_discern_odds = np.asarray([misinfo_discern_odds[i] for i in indices_remap])
misinfo_discern_lb = np.asarray([misinfo_discern_lb[i] for i in indices_remap])
misinfo_discern_ub = np.asarray([misinfo_discern_ub[i] for i in indices_remap])

# plot metrics on Y-axis, odds ratio on X-axis and error bars. Use log scale for X-axis
# plt.errorbar(misinfo_discern_odds, misinfo_discern_metrics_names, xerr=[misinfo_discern_odds - misinfo_discern_lb, misinfo_discern_ub - misinfo_discern_odds], fmt='o', color='black', markerfacecolor='orange', capsize=5)
for i in range(len(misinfo_discern_metrics_names)):
    plt.errorbar(misinfo_discern_metrics_names[i], misinfo_discern_odds[i], yerr=[[misinfo_discern_odds[i] - misinfo_discern_lb[i]], [misinfo_discern_ub[i] - misinfo_discern_odds[i]]], fmt='o', color='black', markerfacecolor=colors[i], capsize=10)

plt.axhline(y=1, color='green', linestyle='--')
plt.yscale('log')
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
# bold the ticks and labels
plt.ylabel('Odds Ratio (Log Scale)', fontsize=12)
plt.xlabel('Metrics', fontsize=12)
plt.xticks(fontsize=11)
# plt.title('Misinformation Discernment vs Odds Ratio')
# plt.show()
# if SAVE_FLAG:
plt.tight_layout()
save_fig('misinfo_discernment_control')
# else:
#     plt.show()

# Intervention effect (misinfo data)

In [ ]:
misinfo_true_correct_path = "results/misinfo_interventions_correct.csv"
misinf_true_correct_unsure_path = "results/misinfo_interventions_correct_unsure.csv"
misinfo_true_trust_path = "results/misinfo_interventions_trust.csv"
misinfo_true_adherence_path = "results/misinfo_interventions_adherence.csv"
misinfo_true_adherence_unsure_path = "results/misinfo_interventions_adherence_unsure.csv"
misinfo_true_unsure_path = "results/misinfo_interventions_unsure.csv"

misinfo_true_correct_df = pd.read_csv(misinfo_true_correct_path)
misinfo_true_correct_unsure_df = pd.read_csv(misinf_true_correct_unsure_path)
misinfo_true_trust_df = pd.read_csv(misinfo_true_trust_path)
misinfo_true_adherence_df = pd.read_csv(misinfo_true_adherence_path)
misinfo_true_adherence_unsure_df = pd.read_csv(misinfo_true_adherence_unsure_path)
misinfo_true_unsure_df = pd.read_csv(misinfo_true_unsure_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_correct_df[misinfo_true_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_correct = [x_axis_interventions_misinfo_true_correct[i] for i in indices_remap]
misinfo_true_correct_df = misinfo_true_correct_df.iloc[indices_remap]
misinfo_true_correct_odds = misinfo_true_correct_df['OR'].apply(lambda x: float(x))
misinfo_true_correct_lb = misinfo_true_correct_df['LB'].apply(lambda x: float(x))
misinfo_true_correct_ub = misinfo_true_correct_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_correct_odds, x_axis_interventions_misinfo_true_correct, xerr=[np.array(misinfo_true_correct_odds) - np.array(misinfo_true_correct_lb), np.array(misinfo_true_correct_ub) - np.array(misinfo_true_correct_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('correct_misinfo_interventions')


x_axis_interventions_misinfo_true_correct_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_correct_unsure_df[misinfo_true_correct_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_correct_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_correct_unsure = [x_axis_interventions_misinfo_true_correct_unsure[i] for i in indices_remap]
misinfo_true_correct_unsure_df = misinfo_true_correct_unsure_df.iloc[indices_remap]
misinfo_true_correct_unsure_odds = misinfo_true_correct_unsure_df['OR'].apply(lambda x: float(x))
misinfo_true_correct_unsure_lb = misinfo_true_correct_unsure_df['LB'].apply(lambda x: float(x))
misinfo_true_correct_unsure_ub = misinfo_true_correct_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_correct_unsure_odds, x_axis_interventions_misinfo_true_correct_unsure, xerr=[np.array(misinfo_true_correct_unsure_odds) - np.array(misinfo_true_correct_unsure_lb), np.array(misinfo_true_correct_unsure_ub) - np.array(misinfo_true_correct_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('correct_unsure_misinfo_interventions')

x_axis_interventions_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_trust_df[misinfo_true_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_trust = [x_axis_interventions_misinfo_true_trust[i] for i in indices_remap]
misinfo_true_trust_df = misinfo_true_trust_df.iloc[indices_remap]
misinfo_true_trust_odds = misinfo_true_trust_df['OR'].apply(lambda x: float(x))
misinfo_true_trust_lb = misinfo_true_trust_df['LB'].apply(lambda x: float(x))
misinfo_true_trust_ub = misinfo_true_trust_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_trust_odds, x_axis_interventions_misinfo_true_trust, xerr=[np.array(misinfo_true_trust_odds) - np.array(misinfo_true_trust_lb), np.array(misinfo_true_trust_ub) - np.array(misinfo_true_trust_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('trust_misinfo_interventions')

x_axis_interventions_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_adherence_df[misinfo_true_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_adherence = [x_axis_interventions_misinfo_true_adherence[i] for i in indices_remap]
misinfo_true_adherence_df = misinfo_true_adherence_df.iloc[indices_remap]
misinfo_true_adherence_odds = misinfo_true_adherence_df['OR'].apply(lambda x: float(x))
misinfo_true_adherence_lb = misinfo_true_adherence_df['LB'].apply(lambda x: float(x))
misinfo_true_adherence_ub = misinfo_true_adherence_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_adherence_odds, x_axis_interventions_misinfo_true_adherence, xerr=[np.array(misinfo_true_adherence_odds) - np.array(misinfo_true_adherence_lb), np.array(misinfo_true_adherence_ub) - np.array(misinfo_true_adherence_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('adherence_misinfo_interventions')

x_axis_interventions_misinfo_true_adherence_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_adherence_unsure_df[misinfo_true_adherence_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_adherence_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_adherence_unsure = [x_axis_interventions_misinfo_true_adherence_unsure[i] for i in indices_remap]
misinfo_true_adherence_unsure_df = misinfo_true_adherence_unsure_df.iloc[indices_remap]
misinfo_true_adherence_unsure_odds = misinfo_true_adherence_unsure_df['OR'].apply(lambda x: float(x))
misinfo_true_adherence_unsure_lb = misinfo_true_adherence_unsure_df['LB'].apply(lambda x: float(x))
misinfo_true_adherence_unsure_ub = misinfo_true_adherence_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_adherence_unsure_odds, x_axis_interventions_misinfo_true_adherence_unsure, xerr=[np.array(misinfo_true_adherence_unsure_odds) - np.array(misinfo_true_adherence_unsure_lb), np.array(misinfo_true_adherence_unsure_ub) - np.array(misinfo_true_adherence_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('adherence_unsure_misinfo_interventions')

x_axis_interventions_misinfo_true_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_true_unsure_df[misinfo_true_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_true_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_true_unsure = [x_axis_interventions_misinfo_true_unsure[i] for i in indices_remap]
misinfo_true_unsure_df = misinfo_true_unsure_df.iloc[indices_remap]
misinfo_true_unsure_odds = misinfo_true_unsure_df['OR'].apply(lambda x: float(x))
misinfo_true_unsure_lb = misinfo_true_unsure_df['LB'].apply(lambda x: float(x))
misinfo_true_unsure_ub = misinfo_true_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_true_unsure_odds, x_axis_interventions_misinfo_true_unsure, xerr=[np.array(misinfo_true_unsure_odds) - np.array(misinfo_true_unsure_lb), np.array(misinfo_true_unsure_ub) - np.array(misinfo_true_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('unsure_misinfo_interventions')

# plot all four metrics in one plot where the lines are spaced out, intervention on X-axis, odds on Y-axis

# set figure size
plt.figure(figsize=(10, 5))
offsets = [-0.3, -0.1, 0.1, 0.3]
x_axis_positions = np.arange(len(intervention_order))

# set significant lb and ub to blue bars
correct_signficant = [is_significant(misinfo_true_correct_lb[i], misinfo_true_correct_ub[i]) for i in range(len(misinfo_true_correct_lb))]
correct_unsure_signficant = [is_significant(misinfo_true_correct_unsure_lb[i], misinfo_true_correct_unsure_ub[i]) for i in range(len(misinfo_true_correct_unsure_lb))]
trust_signficant = [is_significant(misinfo_true_trust_lb[i], misinfo_true_trust_ub[i]) for i in range(len(misinfo_true_trust_lb))]
adherence_signficant = [is_significant(misinfo_true_adherence_lb[i], misinfo_true_adherence_ub[i]) for i in range(len(misinfo_true_adherence_lb))]
adherence_unsure_signficant = [is_significant(misinfo_true_adherence_unsure_lb[i], misinfo_true_adherence_unsure_ub[i]) for i in range(len(misinfo_true_adherence_unsure_lb))]
unsure_signficant = [is_significant(misinfo_true_unsure_lb[i], misinfo_true_unsure_ub[i]) for i in range(len(misinfo_true_unsure_lb))]

colors = ['red', 'blue', 'green', 'orange']  # Marker face colors for different metrics

# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_true_correct_odds, misinfo_true_correct_lb, misinfo_true_correct_ub, 'C1', correct_signficant),
    (misinfo_true_trust_odds, misinfo_true_trust_lb, misinfo_true_trust_ub, 'C4', trust_signficant),
    (misinfo_true_adherence_odds, misinfo_true_adherence_lb, misinfo_true_adherence_ub, 'C5', adherence_signficant),
    (misinfo_true_unsure_odds, misinfo_true_unsure_lb, misinfo_true_unsure_ub, 'C7', unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        plt.errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )



plt.axhline(y=1, color='green', linestyle='--')
plt.xticks(x_axis_positions, intervention_order)
plt.yscale('log')
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Interventions', fontsize=12)
plt.ylabel('Odds Ratio (Log Scale)', fontsize=12)
plt.legend(loc='upper center', fontsize=12, handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C3', label='Trust'), mpatches.Patch(color='C4', label='Adherence'), mpatches.Patch(color='C6', label='Unsure')])
save_fig('misinfo_interventions_all_metrics')

# plot with all metrics including correct+unsure and adherence+unsure
plt.figure(figsize=(14, 4))
offsets = [-0.45, -0.3, -0.15, 0, 0.15, 0.3]
x_axis_positions = np.arange(len(intervention_order))

                               
# Iterate over different metrics


# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_true_correct_odds, misinfo_true_correct_lb, misinfo_true_correct_ub, 'C1', correct_signficant),
    (misinfo_true_correct_unsure_odds, misinfo_true_correct_unsure_lb, misinfo_true_correct_unsure_ub, 'C2', correct_unsure_signficant),
    (misinfo_true_trust_odds, misinfo_true_trust_lb, misinfo_true_trust_ub, 'C4', trust_signficant),
    (misinfo_true_adherence_odds, misinfo_true_adherence_lb, misinfo_true_adherence_ub, 'C5', adherence_signficant),
    (misinfo_true_adherence_unsure_odds, misinfo_true_adherence_unsure_lb, misinfo_true_adherence_unsure_ub, 'C6', adherence_unsure_signficant),
    (misinfo_true_unsure_odds, misinfo_true_unsure_lb, misinfo_true_unsure_ub, 'C7', unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        plt.errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=7, markersize=10
        )

plt.axhline(y=1, color='green', linestyle='--')
plt.xticks(x_axis_positions, intervention_order)
plt.yscale('log')
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Interventions', fontsize=13)
plt.ylabel('Odds Ratio (Log Scale)', fontsize=13)
plt.legend(loc='upper center', handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Accuracy+Unsure'), mpatches.Patch(color='C3', label='Trust'), mpatches.Patch(color='C4', label='Adherence'), mpatches.Patch(color='C5', label='Adherence+Unsure'), mpatches.Patch(color='C6', label='Unsure')], bbox_to_anchor=(0.5, 1.13), ncol=6, fontsize=12)
save_fig('misinfo_interventions_all_metrics_with_unsure')

# Intervention effect (true data)

In [ ]:
misinfo_false_correct_path = "results/no_misinfo_interventions_correct.csv"
misinfo_false_correct_unsure_path = "results/no_misinfo_interventions_correct_unsure.csv"
misinfo_false_trust_path = "results/no_misinfo_interventions_trust.csv"
misinfo_false_adherence_path = "results/no_misinfo_interventions_adherence.csv"
misinfo_false_adherence_unsure_path = "results/no_misinfo_interventions_adherence_unsure.csv"
misinfo_false_unsure_path = "results/no_misinfo_interventions_unsure.csv"

misinfo_false_correct_df = pd.read_csv(misinfo_false_correct_path)
misinfo_false_correct_unsure_df = pd.read_csv(misinfo_false_correct_unsure_path)
misinfo_false_trust_df = pd.read_csv(misinfo_false_trust_path)
misinfo_false_adherence_df = pd.read_csv(misinfo_false_adherence_path)
misinfo_false_adherence_unsure_df = pd.read_csv(misinfo_false_adherence_unsure_path)
misinfo_false_unsure_df = pd.read_csv(misinfo_false_unsure_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_misinfo_false_correct = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_correct_df[misinfo_false_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_correct = [x_axis_interventions_misinfo_false_correct[i] for i in indices_remap]
misinfo_false_correct_df = misinfo_false_correct_df.iloc[indices_remap]
misinfo_false_correct_odds = misinfo_false_correct_df['OR'].apply(lambda x: float(x))
misinfo_false_correct_lb = misinfo_false_correct_df['LB'].apply(lambda x: float(x))
misinfo_false_correct_ub = misinfo_false_correct_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_correct_odds, x_axis_interventions_misinfo_false_correct, xerr=[np.array(misinfo_false_correct_odds) - np.array(misinfo_false_correct_lb), np.array(misinfo_false_correct_ub) - np.array(misinfo_false_correct_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('correct_no_misinfo_interventions')

x_axis_interventions_misinfo_false_correct_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_correct_unsure_df[misinfo_false_correct_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_correct_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_correct_unsure = [x_axis_interventions_misinfo_false_correct_unsure[i] for i in indices_remap]
misinfo_false_correct_unsure_df = misinfo_false_correct_unsure_df.iloc[indices_remap]
misinfo_false_correct_unsure_odds = misinfo_false_correct_unsure_df['OR'].apply(lambda x: float(x))
misinfo_false_correct_unsure_lb = misinfo_false_correct_unsure_df['LB'].apply(lambda x: float(x))
misinfo_false_correct_unsure_ub = misinfo_false_correct_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_correct_unsure_odds, x_axis_interventions_misinfo_false_correct_unsure, xerr=[np.array(misinfo_false_correct_unsure_odds) - np.array(misinfo_false_correct_unsure_lb), np.array(misinfo_false_correct_unsure_ub) - np.array(misinfo_false_correct_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('correct_unsure_no_misinfo_interventions')

x_axis_interventions_misinfo_false_trust = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_trust_df[misinfo_false_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_trust = [x_axis_interventions_misinfo_false_trust[i] for i in indices_remap]
misinfo_false_trust_df = misinfo_false_trust_df.iloc[indices_remap]
misinfo_false_trust_odds = misinfo_false_trust_df['OR'].apply(lambda x: float(x))
misinfo_false_trust_lb = misinfo_false_trust_df['LB'].apply(lambda x: float(x))
misinfo_false_trust_ub = misinfo_false_trust_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_trust_odds, x_axis_interventions_misinfo_false_trust, xerr=[np.array(misinfo_false_trust_odds) - np.array(misinfo_false_trust_lb), np.array(misinfo_false_trust_ub) - np.array(misinfo_false_trust_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('trust_no_misinfo_interventions')

x_axis_interventions_misinfo_false_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_adherence_df[misinfo_false_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_adherence = [x_axis_interventions_misinfo_false_adherence[i] for i in indices_remap]
misinfo_false_adherence_df = misinfo_false_adherence_df.iloc[indices_remap]
misinfo_false_adherence_odds = misinfo_false_adherence_df['OR'].apply(lambda x: float(x))
misinfo_false_adherence_lb = misinfo_false_adherence_df['LB'].apply(lambda x: float(x))
misinfo_false_adherence_ub = misinfo_false_adherence_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_adherence_odds, x_axis_interventions_misinfo_false_adherence, xerr=[np.array(misinfo_false_adherence_odds) - np.array(misinfo_false_adherence_lb), np.array(misinfo_false_adherence_ub) - np.array(misinfo_false_adherence_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('adherence_no_misinfo_interventions')

x_axis_interventions_misinfo_false_adherence_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_adherence_unsure_df[misinfo_false_adherence_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_adherence_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_adherence_unsure = [x_axis_interventions_misinfo_false_adherence_unsure[i] for i in indices_remap]
misinfo_false_adherence_unsure_df = misinfo_false_adherence_unsure_df.iloc[indices_remap]
misinfo_false_adherence_unsure_odds = misinfo_false_adherence_unsure_df['OR'].apply(lambda x: float(x))
misinfo_false_adherence_unsure_lb = misinfo_false_adherence_unsure_df['LB'].apply(lambda x: float(x))
misinfo_false_adherence_unsure_ub = misinfo_false_adherence_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_adherence_unsure_odds, x_axis_interventions_misinfo_false_adherence_unsure, xerr=[np.array(misinfo_false_adherence_unsure_odds) - np.array(misinfo_false_adherence_unsure_lb), np.array(misinfo_false_adherence_unsure_ub) - np.array(misinfo_false_adherence_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('adherence_unsure_no_misinfo_interventions')

x_axis_interventions_misinfo_false_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in misinfo_false_unsure_df[misinfo_false_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_misinfo_false_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_misinfo_false_unsure = [x_axis_interventions_misinfo_false_unsure[i] for i in indices_remap]
misinfo_false_unsure_df = misinfo_false_unsure_df.iloc[indices_remap]
misinfo_false_unsure_odds = misinfo_false_unsure_df['OR'].apply(lambda x: float(x))
misinfo_false_unsure_lb = misinfo_false_unsure_df['LB'].apply(lambda x: float(x))
misinfo_false_unsure_ub = misinfo_false_unsure_df['UB'].apply(lambda x: float(x))

plt.errorbar(misinfo_false_unsure_odds, x_axis_interventions_misinfo_false_unsure, xerr=[np.array(misinfo_false_unsure_odds) - np.array(misinfo_false_unsure_lb), np.array(misinfo_false_unsure_ub) - np.array(misinfo_false_unsure_odds)], fmt='o', color='black', markerfacecolor='orange', capsize=5)
plt.axvline(x=1, color='green', linestyle='--')
plt.xscale('log')
plt.xlabel('Odds Ratio')
plt.ylabel('Interventions')
ax = plt.gca()
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
save_fig('unsure_no_misinfo_interventions')

# plot all four metrics in one plot where the lines are spaced out, intervention on X-axis, odds on Y-axis

# set figure size
plt.figure(figsize=(10, 5))
offsets = [-0.3, -0.1, 0.1, 0.3]
x_axis_positions = np.arange(len(intervention_order))

# set significant lb and ub to blue bars
correct_signficant_no_misinfo = [is_significant(misinfo_false_correct_lb[i], misinfo_false_correct_ub[i]) for i in range(len(misinfo_false_correct_lb))]
correct_unsure_signficant_no_misinfo = [is_significant(misinfo_false_correct_unsure_lb[i], misinfo_false_correct_unsure_ub[i]) for i in range(len(misinfo_false_correct_unsure_lb))]
trust_signficant_no_misinfo = [is_significant(misinfo_false_trust_lb[i], misinfo_false_trust_ub[i]) for i in range(len(misinfo_false_trust_lb))]
adherence_signficant_no_misinfo = [is_significant(misinfo_false_adherence_lb[i], misinfo_false_adherence_ub[i]) for i in range(len(misinfo_false_adherence_lb))]
adherence_unsure_signficant_no_misinfo = [is_significant(misinfo_false_adherence_unsure_lb[i], misinfo_false_adherence_unsure_ub[i]) for i in range(len(misinfo_false_adherence_unsure_lb))]
unsure_signficant_no_misinfo = [is_significant(misinfo_false_unsure_lb[i], misinfo_false_unsure_ub[i]) for i in range(len(misinfo_false_unsure_lb))]

print(correct_signficant_no_misinfo, trust_signficant_no_misinfo, adherence_signficant_no_misinfo, unsure_signficant_no_misinfo)

colors = ['red', 'blue', 'green', 'orange']  # Marker face colors for different metrics

# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_false_correct_odds, misinfo_false_correct_lb, misinfo_false_correct_ub, 'C1', correct_signficant_no_misinfo),
    (misinfo_false_trust_odds, misinfo_false_trust_lb, misinfo_false_trust_ub, 'C4', trust_signficant_no_misinfo),
    (misinfo_false_adherence_odds, misinfo_false_adherence_lb, misinfo_false_adherence_ub, 'C5', adherence_signficant_no_misinfo),
    (misinfo_false_unsure_odds, misinfo_false_unsure_lb, misinfo_false_unsure_ub, 'C7', unsure_signficant_no_misinfo)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        plt.errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )


plt.axhline(y=1, color='green', linestyle='--')
plt.xticks(x_axis_positions, intervention_order)
plt.yscale('log')
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Interventions', fontsize=12)
plt.ylabel('Odds Ratio (Log Scale)', fontsize=12)
plt.legend(loc='upper center', fontsize=12, handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Trust'), mpatches.Patch(color='C3', label='Adherence'), mpatches.Patch(color='C4', label='Unsure')])
save_fig('no_misinfo_interventions_all_metrics')

# plot with all metrics including correct+unsure and adherence+unsure
plt.figure(figsize=(14, 4))
offsets = [-0.45, -0.3, -0.15, 0, 0.15, 0.3]
x_axis_positions = np.arange(len(intervention_order))


# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_false_correct_odds, misinfo_false_correct_lb, misinfo_false_correct_ub, 'C1', correct_signficant_no_misinfo),
    (misinfo_false_correct_unsure_odds, misinfo_false_correct_unsure_lb, misinfo_false_correct_unsure_ub, 'C2', correct_unsure_signficant_no_misinfo),
    (misinfo_false_trust_odds, misinfo_false_trust_lb, misinfo_false_trust_ub, 'C4', trust_signficant_no_misinfo),
    (misinfo_false_adherence_odds, misinfo_false_adherence_lb, misinfo_false_adherence_ub, 'C5', adherence_signficant_no_misinfo),
    (misinfo_false_adherence_unsure_odds, misinfo_false_adherence_unsure_lb, misinfo_false_adherence_unsure_ub, 'C6', adherence_unsure_signficant_no_misinfo),
    (misinfo_false_unsure_odds, misinfo_false_unsure_lb, misinfo_false_unsure_ub, 'C7', unsure_signficant_no_misinfo)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        plt.errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=7, markersize=10
        )

plt.axhline(y=1, color='green', linestyle='--')
plt.xticks(x_axis_positions, intervention_order)
plt.yscale('log')
ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.minorticks_off()
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('Interventions', fontsize=13)
plt.ylabel('Odds Ratio (Log Scale)', fontsize=13)
plt.legend(loc='upper center', handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Accuracy+Unsure'), mpatches.Patch(color='C3', label='Trust'), mpatches.Patch(color='C4', label='Adherence'), mpatches.Patch(color='C5', label='Adherence+Unsure'), mpatches.Patch(color='C6', label='Unsure')], bbox_to_anchor=(0.5, 1.13), ncol=6, fontsize=12)
save_fig('no_misinfo_interventions_all_metrics_with_unsure')



# BreastCancer demography with Misinformation

In [ ]:
breast_cancer_misinfo_true_correct_path = "results/breastcancer_misinfo_interventions_correct.csv"
breast_cancer_misinfo_true_correct_unsure_path = "results/breastcancer_misinfo_interventions_correct_unsure.csv"
breast_cancer_misinfo_true_trust_path = "results/breastcancer_misinfo_interventions_trust.csv"
breast_cancer_misinfo_true_adherence_path = "results/breastcancer_misinfo_interventions_adherence.csv"
breast_cacner_misinfo_true_adherence_unsure_path = "results/breastcancer_misinfo_interventions_adherence_unsure.csv"
breast_cancer_misinfo_true_unsure_path = "results/breastcancer_misinfo_interventions_unsure.csv"

breast_cancer_misinfo_true_correct_df = pd.read_csv(breast_cancer_misinfo_true_correct_path)
breast_cancer_misinfo_true_correct_unsure_df = pd.read_csv(breast_cancer_misinfo_true_correct_unsure_path)
breast_cancer_misinfo_true_trust_df = pd.read_csv(breast_cancer_misinfo_true_trust_path)
breast_cancer_misinfo_true_adherence_df = pd.read_csv(breast_cancer_misinfo_true_adherence_path)
breast_cancer_misinfo_true_adherence_unsure_df = pd.read_csv(breast_cacner_misinfo_true_adherence_unsure_path)
breast_cancer_misinfo_true_unsure_df = pd.read_csv(breast_cancer_misinfo_true_unsure_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_breast_cancer_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_correct_df[breast_cancer_misinfo_true_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_correct = [x_axis_interventions_breast_cancer_misinfo_true_correct[i] for i in indices_remap]
breast_cancer_misinfo_true_correct_df = breast_cancer_misinfo_true_correct_df.iloc[indices_remap]
breast_cancer_misinfo_true_correct_odds = breast_cancer_misinfo_true_correct_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_correct_lb = breast_cancer_misinfo_true_correct_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_correct_ub = breast_cancer_misinfo_true_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_breast_cancer_misinfo_true_correct_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_correct_unsure_df[breast_cancer_misinfo_true_correct_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_correct_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_correct_unsure = [x_axis_interventions_breast_cancer_misinfo_true_correct_unsure[i] for i in indices_remap]
breast_cancer_misinfo_true_correct_unsure_df = breast_cancer_misinfo_true_correct_unsure_df.iloc[indices_remap]
breast_cancer_misinfo_true_correct_unsure_odds = breast_cancer_misinfo_true_correct_unsure_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_correct_unsure_lb = breast_cancer_misinfo_true_correct_unsure_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_correct_unsure_ub = breast_cancer_misinfo_true_correct_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_breast_cancer_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_trust_df[breast_cancer_misinfo_true_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_trust = [x_axis_interventions_breast_cancer_misinfo_true_trust[i] for i in indices_remap]
breast_cancer_misinfo_true_trust_df = breast_cancer_misinfo_true_trust_df.iloc[indices_remap]
breast_cancer_misinfo_true_trust_odds = breast_cancer_misinfo_true_trust_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_trust_lb = breast_cancer_misinfo_true_trust_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_trust_ub = breast_cancer_misinfo_true_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_breast_cancer_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_adherence_df[breast_cancer_misinfo_true_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_adherence = [x_axis_interventions_breast_cancer_misinfo_true_adherence[i] for i in indices_remap]
breast_cancer_misinfo_true_adherence_df = breast_cancer_misinfo_true_adherence_df.iloc[indices_remap]
breast_cancer_misinfo_true_adherence_odds = breast_cancer_misinfo_true_adherence_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_adherence_lb = breast_cancer_misinfo_true_adherence_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_adherence_ub = breast_cancer_misinfo_true_adherence_df['UB'].apply(lambda x: float(x))

x_axis_interventions_breast_cancer_misinfo_true_adherence_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_adherence_unsure_df[breast_cancer_misinfo_true_adherence_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_adherence_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_adherence_unsure = [x_axis_interventions_breast_cancer_misinfo_true_adherence_unsure[i] for i in indices_remap]
breast_cancer_misinfo_true_adherence_unsure_df = breast_cancer_misinfo_true_adherence_unsure_df.iloc[indices_remap]
breast_cancer_misinfo_true_adherence_unsure_odds = breast_cancer_misinfo_true_adherence_unsure_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_adherence_unsure_lb = breast_cancer_misinfo_true_adherence_unsure_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_adherence_unsure_ub = breast_cancer_misinfo_true_adherence_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_breast_cancer_misinfo_true_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in breast_cancer_misinfo_true_unsure_df[breast_cancer_misinfo_true_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_unsure = [x_axis_interventions_breast_cancer_misinfo_true_unsure[i] for i in indices_remap]
breast_cancer_misinfo_true_unsure_df = breast_cancer_misinfo_true_unsure_df.iloc[indices_remap]
breast_cancer_misinfo_true_unsure_odds = breast_cancer_misinfo_true_unsure_df['OR'].apply(lambda x: float(x))
breast_cancer_misinfo_true_unsure_lb = breast_cancer_misinfo_true_unsure_df['LB'].apply(lambda x: float(x))
breast_cancer_misinfo_true_unsure_ub = breast_cancer_misinfo_true_unsure_df['UB'].apply(lambda x: float(x))




# Child demography with Misinformation

In [ ]:
child_misinfo_true_correct_path = "results/children_misinfo_interventions_correct.csv"
child_misinfo_true_correct_unsure_path = "results/children_misinfo_interventions_correct_unsure.csv"
child_misinfo_true_trust_path = "results/children_misinfo_interventions_trust.csv"
child_misinfo_true_adherence_path = "results/children_misinfo_interventions_adherence.csv"
child_misinfo_true_adherence_unsure_path = "results/children_misinfo_interventions_adherence_unsure.csv"
child_misinfo_true_unsure_path = "results/children_misinfo_interventions_unsure.csv"

child_misinfo_true_correct_df = pd.read_csv(child_misinfo_true_correct_path)
child_misinfo_true_correct_unsure_df = pd.read_csv(child_misinfo_true_correct_unsure_path)
child_misinfo_true_trust_df = pd.read_csv(child_misinfo_true_trust_path)
child_misinfo_true_adherence_df = pd.read_csv(child_misinfo_true_adherence_path)
child_misinfo_true_adherence_unsure_df = pd.read_csv(child_misinfo_true_adherence_unsure_path)
child_misinfo_true_unsure_df = pd.read_csv(child_misinfo_true_unsure_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_child_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_correct_df[child_misinfo_true_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_correct = [x_axis_interventions_child_misinfo_true_correct[i] for i in indices_remap]
child_misinfo_true_correct_df = child_misinfo_true_correct_df.iloc[indices_remap]
child_misinfo_true_correct_odds = child_misinfo_true_correct_df['OR'].apply(lambda x: float(x))
child_misinfo_true_correct_lb = child_misinfo_true_correct_df['LB'].apply(lambda x: float(x))
child_misinfo_true_correct_ub = child_misinfo_true_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_correct_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_correct_unsure_df[child_misinfo_true_correct_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_correct_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_correct_unsure = [x_axis_interventions_child_misinfo_true_correct_unsure[i] for i in indices_remap]
child_misinfo_true_correct_unsure_df = child_misinfo_true_correct_unsure_df.iloc[indices_remap]
child_misinfo_true_correct_unsure_odds = child_misinfo_true_correct_unsure_df['OR'].apply(lambda x: float(x))
child_misinfo_true_correct_unsure_lb = child_misinfo_true_correct_unsure_df['LB'].apply(lambda x: float(x))
child_misinfo_true_correct_unsure_ub = child_misinfo_true_correct_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_trust_df[child_misinfo_true_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_trust = [x_axis_interventions_child_misinfo_true_trust[i] for i in indices_remap]
child_misinfo_true_trust_df = child_misinfo_true_trust_df.iloc[indices_remap]
child_misinfo_true_trust_odds = child_misinfo_true_trust_df['OR'].apply(lambda x: float(x))
child_misinfo_true_trust_lb = child_misinfo_true_trust_df['LB'].apply(lambda x: float(x))
child_misinfo_true_trust_ub = child_misinfo_true_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_adherence_df[child_misinfo_true_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_adherence = [x_axis_interventions_child_misinfo_true_adherence[i] for i in indices_remap]
child_misinfo_true_adherence_df = child_misinfo_true_adherence_df.iloc[indices_remap]
child_misinfo_true_adherence_odds = child_misinfo_true_adherence_df['OR'].apply(lambda x: float(x))
child_misinfo_true_adherence_lb = child_misinfo_true_adherence_df['LB'].apply(lambda x: float(x))
child_misinfo_true_adherence_ub = child_misinfo_true_adherence_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_adherence_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_adherence_unsure_df[child_misinfo_true_adherence_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_adherence_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_adherence_unsure = [x_axis_interventions_child_misinfo_true_adherence_unsure[i] for i in indices_remap]
child_misinfo_true_adherence_unsure_df = child_misinfo_true_adherence_unsure_df.iloc[indices_remap]
child_misinfo_true_adherence_unsure_odds = child_misinfo_true_adherence_unsure_df['OR'].apply(lambda x: float(x))
child_misinfo_true_adherence_unsure_lb = child_misinfo_true_adherence_unsure_df['LB'].apply(lambda x: float(x))
child_misinfo_true_adherence_unsure_ub = child_misinfo_true_adherence_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in child_misinfo_true_unsure_df[child_misinfo_true_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_unsure = [x_axis_interventions_child_misinfo_true_unsure[i] for i in indices_remap]
child_misinfo_true_unsure_df = child_misinfo_true_unsure_df.iloc[indices_remap]
child_misinfo_true_unsure_odds = child_misinfo_true_unsure_df['OR'].apply(lambda x: float(x))
child_misinfo_true_unsure_lb = child_misinfo_true_unsure_df['LB'].apply(lambda x: float(x))
child_misinfo_true_unsure_ub = child_misinfo_true_unsure_df['UB'].apply(lambda x: float(x))



# Familiarity demography with Misinformation

In [ ]:
familiarity_misinfo_true_correct_path = "results/familiarity_misinfo_interventions_correct.csv"
familiarity_misinfo_true_correct_unsure_path = "results/familiarity_misinfo_interventions_correct_unsure.csv"
familiarity_misinfo_true_trust_path = "results/familiarity_misinfo_interventions_trust.csv"
familiarity_misinfo_true_adherence_path = "results/familiarity_misinfo_interventions_adherence.csv"
familiarity_misinfo_true_adherence_unsure_path = "results/familiarity_misinfo_interventions_adherence_unsure.csv"
familiarity_misinfo_true_unsure_path = "results/familiarity_misinfo_interventions_unsure.csv"

familiarity_misinfo_true_correct_df = pd.read_csv(familiarity_misinfo_true_correct_path)
familiarity_misinfo_true_correct_unsure_df = pd.read_csv(familiarity_misinfo_true_correct_unsure_path)
familiarity_misinfo_true_trust_df = pd.read_csv(familiarity_misinfo_true_trust_path)
familiarity_misinfo_true_adherence_df = pd.read_csv(familiarity_misinfo_true_adherence_path)
familiarity_misinfo_true_adherence_unsure_df = pd.read_csv(familiarity_misinfo_true_adherence_unsure_path)
familiarity_misinfo_true_unsure_df = pd.read_csv(familiarity_misinfo_true_unsure_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_familiarity_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_correct_df[familiarity_misinfo_true_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_correct = [x_axis_interventions_familiarity_misinfo_true_correct[i] for i in indices_remap]
familiarity_misinfo_true_correct_df = familiarity_misinfo_true_correct_df.iloc[indices_remap]
familiarity_misinfo_true_correct_odds = familiarity_misinfo_true_correct_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_correct_lb = familiarity_misinfo_true_correct_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_correct_ub = familiarity_misinfo_true_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_correct_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_correct_unsure_df[familiarity_misinfo_true_correct_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_correct_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_correct_unsure = [x_axis_interventions_familiarity_misinfo_true_correct_unsure[i] for i in indices_remap]
familiarity_misinfo_true_correct_unsure_df = familiarity_misinfo_true_correct_unsure_df.iloc[indices_remap]
familiarity_misinfo_true_correct_unsure_odds = familiarity_misinfo_true_correct_unsure_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_correct_unsure_lb = familiarity_misinfo_true_correct_unsure_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_correct_unsure_ub = familiarity_misinfo_true_correct_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_trust_df[familiarity_misinfo_true_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_trust = [x_axis_interventions_familiarity_misinfo_true_trust[i] for i in indices_remap]
familiarity_misinfo_true_trust_df = familiarity_misinfo_true_trust_df.iloc[indices_remap]
familiarity_misinfo_true_trust_odds = familiarity_misinfo_true_trust_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_trust_lb = familiarity_misinfo_true_trust_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_trust_ub = familiarity_misinfo_true_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_adherence_df[familiarity_misinfo_true_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_adherence = [x_axis_interventions_familiarity_misinfo_true_adherence[i] for i in indices_remap]
familiarity_misinfo_true_adherence_df = familiarity_misinfo_true_adherence_df.iloc[indices_remap]
familiarity_misinfo_true_adherence_odds = familiarity_misinfo_true_adherence_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_adherence_lb = familiarity_misinfo_true_adherence_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_adherence_ub = familiarity_misinfo_true_adherence_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_adherence_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_adherence_unsure_df[familiarity_misinfo_true_adherence_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_adherence_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_adherence_unsure = [x_axis_interventions_familiarity_misinfo_true_adherence_unsure[i] for i in indices_remap]
familiarity_misinfo_true_adherence_unsure_df = familiarity_misinfo_true_adherence_unsure_df.iloc[indices_remap]
familiarity_misinfo_true_adherence_unsure_odds = familiarity_misinfo_true_adherence_unsure_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_adherence_unsure_lb = familiarity_misinfo_true_adherence_unsure_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_adherence_unsure_ub = familiarity_misinfo_true_adherence_unsure_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_unsure = [str(intervention).replace("Intervention_ID", "") for intervention in familiarity_misinfo_true_unsure_df[familiarity_misinfo_true_unsure_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_unsure.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_unsure = [x_axis_interventions_familiarity_misinfo_true_unsure[i] for i in indices_remap]
familiarity_misinfo_true_unsure_df = familiarity_misinfo_true_unsure_df.iloc[indices_remap]
familiarity_misinfo_true_unsure_odds = familiarity_misinfo_true_unsure_df['OR'].apply(lambda x: float(x))
familiarity_misinfo_true_unsure_lb = familiarity_misinfo_true_unsure_df['LB'].apply(lambda x: float(x))
familiarity_misinfo_true_unsure_ub = familiarity_misinfo_true_unsure_df['UB'].apply(lambda x: float(x))



In [ ]:
# Plot all, breast cancer, children and familiarity in one plot
fig, ax = plt.subplots(4, 1, figsize=(10, 15))
offsets = [-0.3, -0.1, 0.1, 0.3]

# set significant lb and ub to blue bars
breast_cancer_correct_signficant = [is_significant(breast_cancer_misinfo_true_correct_lb[i], breast_cancer_misinfo_true_correct_ub[i]) for i in range(len(breast_cancer_misinfo_true_correct_lb))]
child_correct_signficant = [is_significant(child_misinfo_true_correct_lb[i], child_misinfo_true_correct_ub[i]) for i in range(len(child_misinfo_true_correct_lb))]
familiarity_correct_signficant = [is_significant(familiarity_misinfo_true_correct_lb[i], familiarity_misinfo_true_correct_ub[i]) for i in range(len(familiarity_misinfo_true_correct_lb))]
breast_cancer_correct_unsure_signficant = [is_significant(breast_cancer_misinfo_true_correct_unsure_lb[i], breast_cancer_misinfo_true_correct_unsure_ub[i]) for i in range(len(breast_cancer_misinfo_true_correct_unsure_lb))]
child_correct_unsure_signficant = [is_significant(child_misinfo_true_correct_unsure_lb[i], child_misinfo_true_correct_unsure_ub[i]) for i in range(len(child_misinfo_true_correct_unsure_lb))]
familiarity_correct_unsure_signficant = [is_significant(familiarity_misinfo_true_correct_unsure_lb[i], familiarity_misinfo_true_correct_unsure_ub[i]) for i in range(len(familiarity_misinfo_true_correct_unsure_lb))]
breast_cancer_trust_signficant = [is_significant(breast_cancer_misinfo_true_trust_df['LB'][i], breast_cancer_misinfo_true_trust_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_trust_df['LB']))]
child_trust_signficant = [is_significant(child_misinfo_true_trust_df['LB'][i], child_misinfo_true_trust_df['UB'][i]) for i in range(len(child_misinfo_true_trust_df['LB']))]
familiarity_trust_signficant = [is_significant(familiarity_misinfo_true_trust_df['LB'][i], familiarity_misinfo_true_trust_df['UB'][i]) for i in range(len(familiarity_misinfo_true_trust_df['LB']))]
breast_cancer_adherence_signficant = [is_significant(breast_cancer_misinfo_true_adherence_df['LB'][i], breast_cancer_misinfo_true_adherence_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_adherence_df['LB']))]
child_adherence_signficant = [is_significant(child_misinfo_true_adherence_df['LB'][i], child_misinfo_true_adherence_df['UB'][i]) for i in range(len(child_misinfo_true_adherence_df['LB']))]
familiarity_adherence_signficant = [is_significant(familiarity_misinfo_true_adherence_df['LB'][i], familiarity_misinfo_true_adherence_df['UB'][i]) for i in range(len(familiarity_misinfo_true_adherence_df['LB']))]
breast_cancer_adherence_unsure_signficant = [is_significant(breast_cancer_misinfo_true_adherence_unsure_df['LB'][i], breast_cancer_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_adherence_unsure_df['LB']))]
child_adherence_unsure_signficant = [is_significant(child_misinfo_true_adherence_unsure_df['LB'][i], child_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(child_misinfo_true_adherence_unsure_df['LB']))]
familiarity_adherence_unsure_signficant = [is_significant(familiarity_misinfo_true_adherence_unsure_df['LB'][i], familiarity_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(familiarity_misinfo_true_adherence_unsure_df['LB']))]
breast_cancer_unsure_signficant = [is_significant(breast_cancer_misinfo_true_unsure_df['LB'][i], breast_cancer_misinfo_true_unsure_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_unsure_df['LB']))]
child_unsure_signficant = [is_significant(child_misinfo_true_unsure_df['LB'][i], child_misinfo_true_unsure_df['UB'][i]) for i in range(len(child_misinfo_true_unsure_df['LB']))]
familiarity_unsure_signficant = [is_significant(familiarity_misinfo_true_unsure_df['LB'][i], familiarity_misinfo_true_unsure_df['UB'][i]) for i in range(len(familiarity_misinfo_true_unsure_df['LB']))]

#print all significant values
print("Breast Cancer ", breast_cancer_correct_signficant, breast_cancer_trust_signficant, breast_cancer_adherence_signficant, breast_cancer_unsure_signficant)
print("Children ", child_correct_signficant, child_trust_signficant, child_adherence_signficant, child_unsure_signficant)
print("Familiarity ", familiarity_correct_signficant, familiarity_trust_signficant, familiarity_adherence_signficant, familiarity_unsure_signficant)


colors = ['red', 'blue', 'green', 'orange']  # Marker face colors for different metrics

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_true_correct_odds, misinfo_true_correct_lb, misinfo_true_correct_ub, 'C1', correct_signficant),
    (misinfo_true_trust_odds, misinfo_true_trust_lb, misinfo_true_trust_ub, 'C4', trust_signficant),
    (misinfo_true_adherence_odds, misinfo_true_adherence_lb, misinfo_true_adherence_ub, 'C5', adherence_signficant),
    (misinfo_true_unsure_odds, misinfo_true_unsure_lb, misinfo_true_unsure_ub, 'C7', unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[0].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (breast_cancer_misinfo_true_correct_odds, breast_cancer_misinfo_true_correct_lb, breast_cancer_misinfo_true_correct_ub, 'C1', breast_cancer_correct_signficant),
    (breast_cancer_misinfo_true_trust_odds, breast_cancer_misinfo_true_trust_lb, breast_cancer_misinfo_true_trust_ub, 'C4', breast_cancer_trust_signficant),
    (breast_cancer_misinfo_true_adherence_odds, breast_cancer_misinfo_true_adherence_lb, breast_cancer_misinfo_true_adherence_ub, 'C5', breast_cancer_adherence_signficant),
    (breast_cancer_misinfo_true_unsure_odds, breast_cancer_misinfo_true_unsure_lb, breast_cancer_misinfo_true_unsure_ub, 'C7', breast_cancer_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[1].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (child_misinfo_true_correct_odds, child_misinfo_true_correct_lb, child_misinfo_true_correct_ub, 'C1', child_correct_signficant),
    (child_misinfo_true_trust_odds, child_misinfo_true_trust_lb, child_misinfo_true_trust_ub, 'C2', child_trust_signficant),
    (child_misinfo_true_adherence_odds, child_misinfo_true_adherence_lb, child_misinfo_true_adherence_ub, 'C3', child_adherence_signficant),
    (child_misinfo_true_unsure_odds, child_misinfo_true_unsure_lb, child_misinfo_true_unsure_ub, 'C4', child_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C10' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[2].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (familiarity_misinfo_true_correct_odds, familiarity_misinfo_true_correct_lb, familiarity_misinfo_true_correct_ub, 'C1', familiarity_correct_signficant),
    (familiarity_misinfo_true_trust_odds, familiarity_misinfo_true_trust_lb, familiarity_misinfo_true_trust_ub, 'C4', familiarity_trust_signficant),
    (familiarity_misinfo_true_adherence_odds, familiarity_misinfo_true_adherence_lb, familiarity_misinfo_true_adherence_ub, 'C5', familiarity_adherence_signficant),
    (familiarity_misinfo_true_unsure_odds, familiarity_misinfo_true_unsure_lb, familiarity_misinfo_true_unsure_ub, 'C7', familiarity_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[3].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for a in ax:
    a.axhline(y=1, color='green', linestyle='--')
    a.set_xticks(x_axis_positions)
    a.set_yscale('log')
    a.get_yaxis().set_major_formatter(plt.ScalarFormatter())
    a.minorticks_off()
    a.set_xticklabels(intervention_order)
    a.tick_params(axis='x', labelsize=12)
    a.tick_params(axis='y', labelsize=12)

ax[0].set_ylabel('All', fontsize=12)
ax[1].set_ylabel('Breast Cancer Proximity', fontsize=12)
ax[2].set_ylabel('Has Children', fontsize=12)
ax[3].set_ylabel('Familiarity with AI', fontsize=12)

# set entire figure's super ylabel to Odds Ratio (Log Scale)
fig.supxlabel('Interventions', fontsize=12)
fig.supylabel('Odds Ratio (Log Scale)', fontsize=12)
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4, fontsize=12, handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Trust'), mpatches.Patch(color='C3', label='Adherence'), mpatches.Patch(color='C4', label='Unsure')])

save_fig('misinfo_interventions_all_metrics')

# Alternative figure

In [ ]:
# Plot all, breast cancer, children and familiarity in one plot
fig, ax = plt.subplots(4, 1, figsize=(10, 15))
offsets = [-0.3, -0.1, 0.1, 0.3]

# set significant lb and ub to blue bars
breast_cancer_correct_signficant = [is_significant(breast_cancer_misinfo_true_correct_lb[i], breast_cancer_misinfo_true_correct_ub[i]) for i in range(len(breast_cancer_misinfo_true_correct_lb))]
child_correct_signficant = [is_significant(child_misinfo_true_correct_lb[i], child_misinfo_true_correct_ub[i]) for i in range(len(child_misinfo_true_correct_lb))]
familiarity_correct_signficant = [is_significant(familiarity_misinfo_true_correct_lb[i], familiarity_misinfo_true_correct_ub[i]) for i in range(len(familiarity_misinfo_true_correct_lb))]
breast_cancer_trust_signficant = [is_significant(breast_cancer_misinfo_true_trust_df['LB'][i], breast_cancer_misinfo_true_trust_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_trust_df['LB']))]
child_trust_signficant = [is_significant(child_misinfo_true_trust_df['LB'][i], child_misinfo_true_trust_df['UB'][i]) for i in range(len(child_misinfo_true_trust_df['LB']))]
familiarity_trust_signficant = [is_significant(familiarity_misinfo_true_trust_df['LB'][i], familiarity_misinfo_true_trust_df['UB'][i]) for i in range(len(familiarity_misinfo_true_trust_df['LB']))]
breast_cancer_adherence_signficant = [is_significant(breast_cancer_misinfo_true_adherence_df['LB'][i], breast_cancer_misinfo_true_adherence_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_adherence_df['LB']))]
child_adherence_signficant = [is_significant(child_misinfo_true_adherence_df['LB'][i], child_misinfo_true_adherence_df['UB'][i]) for i in range(len(child_misinfo_true_adherence_df['LB']))]
familiarity_adherence_signficant = [is_significant(familiarity_misinfo_true_adherence_df['LB'][i], familiarity_misinfo_true_adherence_df['UB'][i]) for i in range(len(familiarity_misinfo_true_adherence_df['LB']))]
breast_cancer_unsure_signficant = [is_significant(breast_cancer_misinfo_true_unsure_df['LB'][i], breast_cancer_misinfo_true_unsure_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_unsure_df['LB']))]
child_unsure_signficant = [is_significant(child_misinfo_true_unsure_df['LB'][i], child_misinfo_true_unsure_df['UB'][i]) for i in range(len(child_misinfo_true_unsure_df['LB']))]
familiarity_unsure_signficant = [is_significant(familiarity_misinfo_true_unsure_df['LB'][i], familiarity_misinfo_true_unsure_df['UB'][i]) for i in range(len(familiarity_misinfo_true_unsure_df['LB']))]

colors = ['red', 'blue', 'green', 'orange']  # Marker face colors for different metrics

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_true_correct_odds, misinfo_true_correct_lb, misinfo_true_correct_ub, 'C1', correct_signficant),
    (misinfo_true_trust_odds, misinfo_true_trust_lb, misinfo_true_trust_ub, 'C2', trust_signficant),
    (misinfo_true_adherence_odds, misinfo_true_adherence_lb, misinfo_true_adherence_ub, 'C5', adherence_signficant),
    (misinfo_true_unsure_odds, misinfo_true_unsure_lb, misinfo_true_unsure_ub, 'C7', unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[0].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (breast_cancer_misinfo_true_correct_odds, breast_cancer_misinfo_true_correct_lb, breast_cancer_misinfo_true_correct_ub, 'C1', breast_cancer_correct_signficant),
    (breast_cancer_misinfo_true_trust_odds, breast_cancer_misinfo_true_trust_lb, breast_cancer_misinfo_true_trust_ub, 'C4', breast_cancer_trust_signficant),
    (breast_cancer_misinfo_true_adherence_odds, breast_cancer_misinfo_true_adherence_lb, breast_cancer_misinfo_true_adherence_ub, 'C5', breast_cancer_adherence_signficant),
    (breast_cancer_misinfo_true_unsure_odds, breast_cancer_misinfo_true_unsure_lb, breast_cancer_misinfo_true_unsure_ub, 'C7', breast_cancer_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[1].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (child_misinfo_true_correct_odds, child_misinfo_true_correct_lb, child_misinfo_true_correct_ub, 'C1', child_correct_signficant),
    (child_misinfo_true_trust_odds, child_misinfo_true_trust_lb, child_misinfo_true_trust_ub, 'C4', child_trust_signficant),
    (child_misinfo_true_adherence_odds, child_misinfo_true_adherence_lb, child_misinfo_true_adherence_ub, 'C5', child_adherence_signficant),
    (child_misinfo_true_unsure_odds, child_misinfo_true_unsure_lb, child_misinfo_true_unsure_ub, 'C7', child_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[2].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (familiarity_misinfo_true_correct_odds, familiarity_misinfo_true_correct_lb, familiarity_misinfo_true_correct_ub, 'C1', familiarity_correct_signficant),
    (familiarity_misinfo_true_trust_odds, familiarity_misinfo_true_trust_lb, familiarity_misinfo_true_trust_ub, 'C4', familiarity_trust_signficant),
    (familiarity_misinfo_true_adherence_odds, familiarity_misinfo_true_adherence_lb, familiarity_misinfo_true_adherence_ub, 'C5', familiarity_adherence_signficant),
    (familiarity_misinfo_true_unsure_odds, familiarity_misinfo_true_unsure_lb, familiarity_misinfo_true_unsure_ub, 'C7', familiarity_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[3].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5
        )

for a in ax:
    a.axhline(y=1, color='green', linestyle='--')
    a.set_xticks(x_axis_positions)
    a.set_yscale('log')
    a.get_yaxis().set_major_formatter(plt.ScalarFormatter())
    a.minorticks_off()
    a.set_xticklabels(intervention_order)
    a.tick_params(axis='x', labelsize=12)
    a.tick_params(axis='y', labelsize=12)

# ax[0].set_ylabel('All', fontsize=12)
# ax[1].set_ylabel('Breast Cancer Proximity', fontsize=12)
# ax[2].set_ylabel('Has Children', fontsize=12)
# ax[3].set_ylabel('Familiarity with AI', fontsize=12)

# Set captions to each subfigure
ax[0].set_title(r'\textbf{a) All participants}', fontsize=12, pad=-14, y=1)
ax[1].set_title(r'\textbf{b) Breast cancer proximity participants}', fontsize=12, pad=-14, y=1)
ax[2].set_title(r'\textbf{c) Participants with children}', fontsize=12, pad=-14, y=1)
ax[3].set_title(r'\textbf{d) Familiarity with AI}', fontsize=12, pad=-14, y=1)

# set entire figure's super ylabel to Odds Ratio (Log Scale)
fig.supxlabel('Interventions', fontsize=13)
fig.supylabel('Odds Ratio (Log Scale)', fontsize=13)
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=4, fontsize=12, handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Trust'), mpatches.Patch(color='C3', label='Adherence'), mpatches.Patch(color='C4', label='Unsure')])

save_fig('misinfo_interventions_all_metrics_alternative')

In [ ]:
# Plot all, breast cancer, children and familiarity in one plot
fig, ax = plt.subplots(4, 1, figsize=(14, 15))
offsets = [-0.45, -0.3, -0.15, 0, 0.15, 0.3]
lines = ['solid', 'dashed', 'dotted', 'dashdot']
markers = ['o', 'D', 'H', 's']

# set significant lb and ub to blue bars
breast_cancer_correct_signficant = [is_significant(breast_cancer_misinfo_true_correct_lb[i], breast_cancer_misinfo_true_correct_ub[i]) for i in range(len(breast_cancer_misinfo_true_correct_lb))]
child_correct_signficant = [is_significant(child_misinfo_true_correct_lb[i], child_misinfo_true_correct_ub[i]) for i in range(len(child_misinfo_true_correct_lb))]
familiarity_correct_signficant = [is_significant(familiarity_misinfo_true_correct_lb[i], familiarity_misinfo_true_correct_ub[i]) for i in range(len(familiarity_misinfo_true_correct_lb))]
breast_cancer_correct_unsure_signficant = [is_significant(breast_cancer_misinfo_true_correct_unsure_lb[i], breast_cancer_misinfo_true_correct_unsure_ub[i]) for i in range(len(breast_cancer_misinfo_true_correct_unsure_lb))]
child_correct_unsure_signficant = [is_significant(child_misinfo_true_correct_unsure_lb[i], child_misinfo_true_correct_unsure_ub[i]) for i in range(len(child_misinfo_true_correct_unsure_lb))]
familiarity_correct_unsure_signficant = [is_significant(familiarity_misinfo_true_correct_unsure_lb[i], familiarity_misinfo_true_correct_unsure_ub[i]) for i in range(len(familiarity_misinfo_true_correct_unsure_lb))]
breast_cancer_trust_signficant = [is_significant(breast_cancer_misinfo_true_trust_df['LB'][i], breast_cancer_misinfo_true_trust_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_trust_df['LB']))]
child_trust_signficant = [is_significant(child_misinfo_true_trust_df['LB'][i], child_misinfo_true_trust_df['UB'][i]) for i in range(len(child_misinfo_true_trust_df['LB']))]
familiarity_trust_signficant = [is_significant(familiarity_misinfo_true_trust_df['LB'][i], familiarity_misinfo_true_trust_df['UB'][i]) for i in range(len(familiarity_misinfo_true_trust_df['LB']))]
breast_cancer_adherence_signficant = [is_significant(breast_cancer_misinfo_true_adherence_df['LB'][i], breast_cancer_misinfo_true_adherence_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_adherence_df['LB']))]
child_adherence_signficant = [is_significant(child_misinfo_true_adherence_df['LB'][i], child_misinfo_true_adherence_df['UB'][i]) for i in range(len(child_misinfo_true_adherence_df['LB']))]
familiarity_adherence_signficant = [is_significant(familiarity_misinfo_true_adherence_df['LB'][i], familiarity_misinfo_true_adherence_df['UB'][i]) for i in range(len(familiarity_misinfo_true_adherence_df['LB']))]
breast_cancer_adherence_unsure_signficant = [is_significant(breast_cancer_misinfo_true_adherence_unsure_df['LB'][i], breast_cancer_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_adherence_unsure_df['LB']))]
child_adherence_unsure_signficant = [is_significant(child_misinfo_true_adherence_unsure_df['LB'][i], child_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(child_misinfo_true_adherence_unsure_df['LB']))]
familiarity_adherence_unsure_signficant = [is_significant(familiarity_misinfo_true_adherence_unsure_df['LB'][i], familiarity_misinfo_true_adherence_unsure_df['UB'][i]) for i in range(len(familiarity_misinfo_true_adherence_unsure_df['LB']))]
breast_cancer_unsure_signficant = [is_significant(breast_cancer_misinfo_true_unsure_df['LB'][i], breast_cancer_misinfo_true_unsure_df['UB'][i]) for i in range(len(breast_cancer_misinfo_true_unsure_df['LB']))]
child_unsure_signficant = [is_significant(child_misinfo_true_unsure_df['LB'][i], child_misinfo_true_unsure_df['UB'][i]) for i in range(len(child_misinfo_true_unsure_df['LB']))]
familiarity_unsure_signficant = [is_significant(familiarity_misinfo_true_unsure_df['LB'][i], familiarity_misinfo_true_unsure_df['UB'][i]) for i in range(len(familiarity_misinfo_true_unsure_df['LB']))]


for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (misinfo_true_correct_odds, misinfo_true_correct_lb, misinfo_true_correct_ub, 'C1', correct_signficant),
    (misinfo_true_correct_unsure_odds, misinfo_true_correct_unsure_lb, misinfo_true_correct_unsure_ub, 'C2', correct_unsure_signficant),
    (misinfo_true_trust_odds, misinfo_true_trust_lb, misinfo_true_trust_ub, 'C4', trust_signficant),
    (misinfo_true_adherence_odds, misinfo_true_adherence_lb, misinfo_true_adherence_ub, 'C5', adherence_signficant),
    (misinfo_true_adherence_unsure_odds, misinfo_true_adherence_unsure_lb, misinfo_true_adherence_unsure_ub, 'C6', adherence_unsure_signficant),
    (misinfo_true_unsure_odds, misinfo_true_unsure_lb, misinfo_true_unsure_ub, 'C7', unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[0].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt=markers[j], color=color, markerfacecolor=marker_color, capsize=5, markersize=10
        )

# Iterate over different metrics
for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (breast_cancer_misinfo_true_correct_odds, breast_cancer_misinfo_true_correct_lb, breast_cancer_misinfo_true_correct_ub, 'C1', breast_cancer_correct_signficant),
    (breast_cancer_misinfo_true_correct_unsure_odds, breast_cancer_misinfo_true_correct_unsure_lb, breast_cancer_misinfo_true_correct_unsure_ub, 'C2', breast_cancer_correct_unsure_signficant),
    (breast_cancer_misinfo_true_trust_odds, breast_cancer_misinfo_true_trust_lb, breast_cancer_misinfo_true_trust_ub, 'C4', breast_cancer_trust_signficant),
    (breast_cancer_misinfo_true_adherence_odds, breast_cancer_misinfo_true_adherence_lb, breast_cancer_misinfo_true_adherence_ub, 'C5', breast_cancer_adherence_signficant),
    (breast_cancer_misinfo_true_adherence_unsure_odds, breast_cancer_misinfo_true_adherence_unsure_lb, breast_cancer_misinfo_true_adherence_unsure_ub, 'C6', breast_cancer_adherence_unsure_signficant),
    (breast_cancer_misinfo_true_unsure_odds, breast_cancer_misinfo_true_unsure_lb, breast_cancer_misinfo_true_unsure_ub, 'C7', breast_cancer_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        # clip max y error values to -10 to 10
        # odds[j] = np.clip(odds[j], -10, 10)
        lb[j] = np.clip(lb[j], -2, 10)
        ub[j] = np.clip(ub[j], -10, 10)
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[3].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt=markers[j], color=color, markerfacecolor=marker_color, capsize=5, markersize=10
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (child_misinfo_true_correct_odds, child_misinfo_true_correct_lb, child_misinfo_true_correct_ub, 'C1', child_correct_signficant),
    (child_misinfo_true_correct_unsure_odds, child_misinfo_true_correct_unsure_lb, child_misinfo_true_correct_unsure_ub, 'C2', child_correct_unsure_signficant),
    (child_misinfo_true_trust_odds, child_misinfo_true_trust_lb, child_misinfo_true_trust_ub, 'C4', child_trust_signficant),
    (child_misinfo_true_adherence_odds, child_misinfo_true_adherence_lb, child_misinfo_true_adherence_ub, 'C5', child_adherence_signficant),
    (child_misinfo_true_adherence_unsure_odds, child_misinfo_true_adherence_unsure_lb, child_misinfo_true_adherence_unsure_ub, 'C6', child_adherence_unsure_signficant),
    (child_misinfo_true_unsure_odds, child_misinfo_true_unsure_lb, child_misinfo_true_unsure_ub, 'C7', child_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[2].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt=markers[j], color=color, markerfacecolor=marker_color, capsize=5, markersize=10
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (familiarity_misinfo_true_correct_odds, familiarity_misinfo_true_correct_lb, familiarity_misinfo_true_correct_ub, 'C1', familiarity_correct_signficant),
    (familiarity_misinfo_true_correct_unsure_odds, familiarity_misinfo_true_correct_unsure_lb, familiarity_misinfo_true_correct_unsure_ub, 'C2', familiarity_correct_unsure_signficant),
    (familiarity_misinfo_true_trust_odds, familiarity_misinfo_true_trust_lb, familiarity_misinfo_true_trust_ub, 'C4', familiarity_trust_signficant),
    (familiarity_misinfo_true_adherence_odds, familiarity_misinfo_true_adherence_lb, familiarity_misinfo_true_adherence_ub, 'C5', familiarity_adherence_signficant),
    (familiarity_misinfo_true_adherence_unsure_odds, familiarity_misinfo_true_adherence_unsure_lb, familiarity_misinfo_true_adherence_unsure_ub, 'C6', familiarity_adherence_unsure_signficant),
    (familiarity_misinfo_true_unsure_odds, familiarity_misinfo_true_unsure_lb, familiarity_misinfo_true_unsure_ub, 'C7', familiarity_unsure_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C3' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[1].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt=markers[j], color=color, markerfacecolor=marker_color, capsize=5, markersize=10
        )

for a in ax:
    a.axhline(y=1, color='green', linestyle='--')
    a.set_xticks(x_axis_positions)
    a.set_yscale('log')
    a.get_yaxis().set_major_formatter(plt.ScalarFormatter())
    a.minorticks_off()
    a.set_xticklabels(intervention_order)
    a.tick_params(axis='x', labelsize=12)
    a.tick_params(axis='y', labelsize=12)

# ax[0].set_ylabel('All', fontsize=12)
# ax[1].set_ylabel('Breast Cancer Proximity', fontsize=12)
# ax[2].set_ylabel('Has Children', fontsize=12)
# ax[3].set_ylabel('Familiarity with AI', fontsize=12)

# Set captions to each subfigure
ax[0].set_title(r'\textbf{a) All participants}', fontsize=12, pad=-14, y=1)
ax[3].set_title(r'\textbf{d) Breast cancer proximity participants}', fontsize=12, pad=-14, y=1)
ax[2].set_title(r'\textbf{c) Participants with children}', fontsize=12, pad=-14, y=1)
ax[1].set_title(r'\textbf{b) Familiarity with AI}', fontsize=12, pad=-14, y=1)

# set entire figure's super ylabel to Odds Ratio (Log Scale)
fig.supxlabel('Interventions', fontsize=13)
fig.supylabel('Odds Ratio (Log Scale)', fontsize=13)
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.03), ncol=6, fontsize=12, handles=[mpatches.Patch(color='C1', label='Strict Accuracy'), mpatches.Patch(color='C2', label='Lenient Accuracy'), mpatches.Patch(color='C4', label='Trust'), mpatches.Patch(color='C5', label='Strict Adherence'), mpatches.Patch(color='C6', label='Lenient Adherence'), mpatches.Patch(color='C7', label='Unsure')])

save_fig('misinfo_interventions_all_metrics_alternative_with_unsure')

# Strictness condition breast cancer misinfo data

In [ ]:
strict_misinfo_true_breast_cancer_correct_path = "results/strict_breastcancer_misinfo_interventions_correct.csv"
strict_misinfo_true_breast_cancer_trust_path = "results/strict_breastcancer_misinfo_interventions_trust.csv"
strict_misinfo_true_breast_cancer_adherence_path = "results/strict_breastcancer_misinfo_interventions_adherence.csv"

strict_misinfo_true_breast_cancer_correct_df = pd.read_csv(strict_misinfo_true_breast_cancer_correct_path)
strict_misinfo_true_breast_cancer_trust_df = pd.read_csv(strict_misinfo_true_breast_cancer_trust_path)
strict_misinfo_true_breast_cancer_adherence_df = pd.read_csv(strict_misinfo_true_breast_cancer_adherence_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_breast_cancer_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_breast_cancer_correct_df[strict_misinfo_true_breast_cancer_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_correct = [x_axis_interventions_breast_cancer_misinfo_true_correct[i] for i in indices_remap]
strict_misinfo_true_breast_cancer_correct_df = strict_misinfo_true_breast_cancer_correct_df.iloc[indices_remap]
strict_misinfo_true_breast_cancer_correct_odds = strict_misinfo_true_breast_cancer_correct_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_correct_lb = strict_misinfo_true_breast_cancer_correct_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_correct_ub = strict_misinfo_true_breast_cancer_correct_df['UB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_correct_df = strict_misinfo_true_breast_cancer_correct_df.iloc[indices_remap]

x_axis_interventions_breast_cancer_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_breast_cancer_trust_df[strict_misinfo_true_breast_cancer_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_trust = [x_axis_interventions_breast_cancer_misinfo_true_trust[i] for i in indices_remap]
strict_misinfo_true_breast_cancer_trust_df = strict_misinfo_true_breast_cancer_trust_df.iloc[indices_remap]
strict_misinfo_true_breast_cancer_trust_odds = strict_misinfo_true_breast_cancer_trust_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_trust_lb = strict_misinfo_true_breast_cancer_trust_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_trust_ub = strict_misinfo_true_breast_cancer_trust_df['UB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_trust_df = strict_misinfo_true_breast_cancer_trust_df.iloc[indices_remap]

x_axis_interventions_breast_cancer_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_breast_cancer_adherence_df[strict_misinfo_true_breast_cancer_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_breast_cancer_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_breast_cancer_misinfo_true_adherence = [x_axis_interventions_breast_cancer_misinfo_true_adherence[i] for i in indices_remap]
strict_misinfo_true_breast_cancer_adherence_df = strict_misinfo_true_breast_cancer_adherence_df.iloc[indices_remap]
strict_misinfo_true_breast_cancer_adherence_odds = strict_misinfo_true_breast_cancer_adherence_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_adherence_lb = strict_misinfo_true_breast_cancer_adherence_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_adherence_ub = strict_misinfo_true_breast_cancer_adherence_df['UB'].apply(lambda x: float(x))
strict_misinfo_true_breast_cancer_adherence_df = strict_misinfo_true_breast_cancer_adherence_df.iloc[indices_remap]




# Strictness condition child misinfo data

In [ ]:
strict_misinfo_true_child_correct_path = "results/strict_children_misinfo_interventions_correct.csv"
strict_misinfo_true_child_trust_path = "results/strict_children_misinfo_interventions_trust.csv"
strict_misinfo_true_child_adherence_path = "results/strict_children_misinfo_interventions_adherence.csv"

strict_misinfo_true_child_correct_df = pd.read_csv(strict_misinfo_true_child_correct_path)
strict_misinfo_true_child_trust_df = pd.read_csv(strict_misinfo_true_child_trust_path)
strict_misinfo_true_child_adherence_df = pd.read_csv(strict_misinfo_true_child_adherence_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_child_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_child_correct_df[strict_misinfo_true_child_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_correct = [x_axis_interventions_child_misinfo_true_correct[i] for i in indices_remap]
strict_misinfo_true_child_correct_df = strict_misinfo_true_child_correct_df.iloc[indices_remap]
strict_misinfo_true_child_correct_odds = strict_misinfo_true_child_correct_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_child_correct_lb = strict_misinfo_true_child_correct_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_child_correct_ub = strict_misinfo_true_child_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_child_trust_df[strict_misinfo_true_child_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_trust = [x_axis_interventions_child_misinfo_true_trust[i] for i in indices_remap]
strict_misinfo_true_child_trust_df = strict_misinfo_true_child_trust_df.iloc[indices_remap]
strict_misinfo_true_child_trust_odds = strict_misinfo_true_child_trust_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_child_trust_lb = strict_misinfo_true_child_trust_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_child_trust_ub = strict_misinfo_true_child_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_child_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_child_adherence_df[strict_misinfo_true_child_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_child_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_child_misinfo_true_adherence = [x_axis_interventions_child_misinfo_true_adherence[i] for i in indices_remap]
strict_misinfo_true_child_adherence_df = strict_misinfo_true_child_adherence_df.iloc[indices_remap]
strict_misinfo_true_child_adherence_odds = strict_misinfo_true_child_adherence_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_child_adherence_lb = strict_misinfo_true_child_adherence_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_child_adherence_ub = strict_misinfo_true_child_adherence_df['UB'].apply(lambda x: float(x))


# Strictness condition familiarity misinfo data

In [ ]:
strict_misinfo_true_familiarity_correct_path = "results/strict_familiarity_misinfo_interventions_correct.csv"
strict_misinfo_true_familiarity_trust_path = "results/strict_familiarity_misinfo_interventions_trust.csv"
strict_misinfo_true_familiarity_adherence_path = "results/strict_familiarity_misinfo_interventions_adherence.csv"

strict_misinfo_true_familiarity_correct_df = pd.read_csv(strict_misinfo_true_familiarity_correct_path)
strict_misinfo_true_familiarity_trust_df = pd.read_csv(strict_misinfo_true_familiarity_trust_path)
strict_misinfo_true_familiarity_adherence_df = pd.read_csv(strict_misinfo_true_familiarity_adherence_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_familiarity_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_familiarity_correct_df[strict_misinfo_true_familiarity_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_correct = [x_axis_interventions_familiarity_misinfo_true_correct[i] for i in indices_remap]
strict_misinfo_true_familiarity_correct_df = strict_misinfo_true_familiarity_correct_df.iloc[indices_remap]
strict_misinfo_true_familiarity_correct_odds = strict_misinfo_true_familiarity_correct_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_correct_lb = strict_misinfo_true_familiarity_correct_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_correct_ub = strict_misinfo_true_familiarity_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_familiarity_trust_df[strict_misinfo_true_familiarity_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_trust = [x_axis_interventions_familiarity_misinfo_true_trust[i] for i in indices_remap]
strict_misinfo_true_familiarity_trust_df = strict_misinfo_true_familiarity_trust_df.iloc[indices_remap]
strict_misinfo_true_familiarity_trust_odds = strict_misinfo_true_familiarity_trust_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_trust_lb = strict_misinfo_true_familiarity_trust_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_trust_ub = strict_misinfo_true_familiarity_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_familiarity_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_familiarity_adherence_df[strict_misinfo_true_familiarity_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_familiarity_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_familiarity_misinfo_true_adherence = [x_axis_interventions_familiarity_misinfo_true_adherence[i] for i in indices_remap]
strict_misinfo_true_familiarity_adherence_df = strict_misinfo_true_familiarity_adherence_df.iloc[indices_remap]
strict_misinfo_true_familiarity_adherence_odds = strict_misinfo_true_familiarity_adherence_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_adherence_lb = strict_misinfo_true_familiarity_adherence_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_familiarity_adherence_ub = strict_misinfo_true_familiarity_adherence_df['UB'].apply(lambda x: float(x))


# Strict all demographics misinfo data

In [ ]:
strict_misinfo_true_correct_path = "results/strict_misinfo_interventions_correct.csv"
strict_misinfo_true_trust_path = "results/strict_misinfo_interventions_trust.csv"
strict_misinfo_true_adherence_path = "results/strict_misinfo_interventions_adherence.csv"

strict_misinfo_true_correct_df = pd.read_csv(strict_misinfo_true_correct_path)
strict_misinfo_true_trust_df = pd.read_csv(strict_misinfo_true_trust_path)
strict_misinfo_true_adherence_df = pd.read_csv(strict_misinfo_true_adherence_path)

intervention_order = ['Pop-up', 'In-line', 'RAG', 'Priming']
x_axis_interventions_strict_misinfo_true_correct = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_correct_df[strict_misinfo_true_correct_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_strict_misinfo_true_correct.index(intervention) for intervention in intervention_order]
x_axis_interventions_strict_misinfo_true_correct = [x_axis_interventions_strict_misinfo_true_correct[i] for i in indices_remap]
strict_misinfo_true_correct_df = strict_misinfo_true_correct_df.iloc[indices_remap]
strict_misinfo_true_correct_odds = strict_misinfo_true_correct_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_correct_lb = strict_misinfo_true_correct_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_correct_ub = strict_misinfo_true_correct_df['UB'].apply(lambda x: float(x))

x_axis_interventions_strict_misinfo_true_trust = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_trust_df[strict_misinfo_true_trust_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_strict_misinfo_true_trust.index(intervention) for intervention in intervention_order]
x_axis_interventions_strict_misinfo_true_trust = [x_axis_interventions_strict_misinfo_true_trust[i] for i in indices_remap]
strict_misinfo_true_trust_df = strict_misinfo_true_trust_df.iloc[indices_remap]
strict_misinfo_true_trust_odds = strict_misinfo_true_trust_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_trust_lb = strict_misinfo_true_trust_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_trust_ub = strict_misinfo_true_trust_df['UB'].apply(lambda x: float(x))

x_axis_interventions_strict_misinfo_true_adherence = [str(intervention).replace("Intervention_ID", "") for intervention in strict_misinfo_true_adherence_df[strict_misinfo_true_adherence_df.columns[0]].tolist()]
indices_remap = [x_axis_interventions_strict_misinfo_true_adherence.index(intervention) for intervention in intervention_order]
x_axis_interventions_strict_misinfo_true_adherence = [x_axis_interventions_strict_misinfo_true_adherence[i] for i in indices_remap]
strict_misinfo_true_adherence_df = strict_misinfo_true_adherence_df.iloc[indices_remap]
strict_misinfo_true_adherence_odds = strict_misinfo_true_adherence_df['OR'].apply(lambda x: float(x))
strict_misinfo_true_adherence_lb = strict_misinfo_true_adherence_df['LB'].apply(lambda x: float(x))
strict_misinfo_true_adherence_ub = strict_misinfo_true_adherence_df['UB'].apply(lambda x: float(x))



In [ ]:
# plot strict breast cancer, children and familiarity in one plot
fig, ax = plt.subplots(4, 1, figsize=(10, 15))
offsets = [-0.3, -0.1, 0.1]

# four different lines for each intervention
lines = ['solid', 'dashed', 'dashdot', 'dotted']

# set significant lb and ub to blue bars
strict_all_correct_signficant = [is_significant(strict_misinfo_true_correct_lb[i], strict_misinfo_true_correct_ub[i]) for i in range(len(strict_misinfo_true_correct_lb))]
strict_all_trust_signficant = [is_significant(strict_misinfo_true_trust_lb[i], strict_misinfo_true_trust_ub[i]) for i in range(len(strict_misinfo_true_trust_lb))]
strict_all_adherence_signficant = [is_significant(strict_misinfo_true_adherence_lb[i], strict_misinfo_true_adherence_ub[i]) for i in range(len(strict_misinfo_true_adherence_lb))]
strict_breast_cancer_correct_signficant = [is_significant(strict_misinfo_true_breast_cancer_correct_lb[i], strict_misinfo_true_breast_cancer_correct_ub[i]) for i in range(len(strict_misinfo_true_breast_cancer_correct_lb))]
strict_breast_cancer_trust_signficant = [is_significant(strict_misinfo_true_breast_cancer_trust_lb[i], strict_misinfo_true_breast_cancer_trust_ub[i]) for i in range(len(strict_misinfo_true_breast_cancer_trust_lb))]
strict_breast_cancer_adherence_signficant = [is_significant(strict_misinfo_true_breast_cancer_adherence_lb[i], strict_misinfo_true_breast_cancer_adherence_ub[i]) for i in range(len(strict_misinfo_true_breast_cancer_adherence_lb))]
strict_child_correct_signficant = [is_significant(strict_misinfo_true_child_correct_lb[i], strict_misinfo_true_child_correct_ub[i]) for i in range(len(strict_misinfo_true_child_correct_lb))]
strict_child_trust_signficant = [is_significant(strict_misinfo_true_child_trust_lb[i], strict_misinfo_true_child_trust_ub[i]) for i in range(len(strict_misinfo_true_child_trust_lb))]
strict_child_adherence_signficant = [is_significant(strict_misinfo_true_child_adherence_lb[i], strict_misinfo_true_child_adherence_ub[i]) for i in range(len(strict_misinfo_true_child_adherence_lb))]
strict_familiarity_correct_signficant = [is_significant(strict_misinfo_true_familiarity_correct_lb[i], strict_misinfo_true_familiarity_correct_ub[i]) for i in range(len(strict_misinfo_true_familiarity_correct_lb))]
strict_familiarity_trust_signficant = [is_significant(strict_misinfo_true_familiarity_trust_lb[i], strict_misinfo_true_familiarity_trust_ub[i]) for i in range(len(strict_misinfo_true_familiarity_trust_lb))]
strict_familiarity_adherence_signficant = [is_significant(strict_misinfo_true_familiarity_adherence_lb[i], strict_misinfo_true_familiarity_adherence_ub[i]) for i in range(len(strict_misinfo_true_familiarity_adherence_lb))]

colors = ['red', 'blue', 'green', 'orange']  # Marker face colors for different metrics

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (strict_misinfo_true_correct_odds, strict_misinfo_true_correct_lb, strict_misinfo_true_correct_ub, 'C1', strict_all_correct_signficant),
    (strict_misinfo_true_trust_odds, strict_misinfo_true_trust_lb, strict_misinfo_true_trust_ub, 'C2', strict_all_trust_signficant),
    (strict_misinfo_true_adherence_odds, strict_misinfo_true_adherence_lb, strict_misinfo_true_adherence_ub, 'C3', strict_all_adherence_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C10' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[0].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5, linestyle=lines[0]
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (strict_misinfo_true_breast_cancer_correct_odds, strict_misinfo_true_breast_cancer_correct_lb, strict_misinfo_true_breast_cancer_correct_ub, 'C1', strict_breast_cancer_correct_signficant),
    (strict_misinfo_true_breast_cancer_trust_odds, strict_misinfo_true_breast_cancer_trust_lb, strict_misinfo_true_breast_cancer_trust_ub, 'C2', strict_breast_cancer_trust_signficant),
    (strict_misinfo_true_breast_cancer_adherence_odds, strict_misinfo_true_breast_cancer_adherence_lb, strict_misinfo_true_breast_cancer_adherence_ub, 'C3', strict_breast_cancer_adherence_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C10' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        # clip y error values to -10 to 10
        lb[j] = max(lb[j], -10)
        ub[j] = min(ub[j], 10)
        ax[3].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5,linestyle=lines[1]
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (strict_misinfo_true_child_correct_odds, strict_misinfo_true_child_correct_lb, strict_misinfo_true_child_correct_ub, 'C1', strict_child_correct_signficant),
    (strict_misinfo_true_child_trust_odds, strict_misinfo_true_child_trust_lb, strict_misinfo_true_child_trust_ub, 'C2', strict_child_trust_signficant),
    (strict_misinfo_true_child_adherence_odds, strict_misinfo_true_child_adherence_lb, strict_misinfo_true_child_adherence_ub, 'C3', strict_child_adherence_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C10' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[2].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5, linestyle=lines[2]
        )

for i, (odds, lb, ub, marker_color, binary_mask) in enumerate([
    (strict_misinfo_true_familiarity_correct_odds, strict_misinfo_true_familiarity_correct_lb, strict_misinfo_true_familiarity_correct_ub, 'C1', strict_familiarity_correct_signficant),
    (strict_misinfo_true_familiarity_trust_odds, strict_misinfo_true_familiarity_trust_lb, strict_misinfo_true_familiarity_trust_ub, 'C2', strict_familiarity_trust_signficant),
    (strict_misinfo_true_familiarity_adherence_odds, strict_misinfo_true_familiarity_adherence_lb, strict_misinfo_true_familiarity_adherence_ub, 'C3', strict_familiarity_adherence_signficant)
]):
    for j, x in enumerate(x_axis_positions):
        color = 'C10' if binary_mask[j] == 1 else 'black'  # Set color based on binary mask
        ax[1].errorbar(
            x + offsets[i], odds[j],
            yerr=[[odds[j] - lb[j]], [ub[j] - odds[j]]],
            fmt='o', color=color, markerfacecolor=marker_color, capsize=5, linestyle=lines[3]
        )
        

for a in ax:
    a.axhline(y=1, color='green', linestyle='--')
    a.set_xticks(x_axis_positions)
    a.set_yscale('log')
    a.get_yaxis().set_major_formatter(plt.ScalarFormatter())
    a.minorticks_off()
    a.set_xticklabels(intervention_order)
    a.tick_params(axis='x', labelsize=12)
    a.tick_params(axis='y', labelsize=12)

ax[0].set_ylabel('All', fontsize=12)
ax[3].set_ylabel('Breast Cancer Proximity', fontsize=12)
ax[2].set_ylabel('Has Children', fontsize=12)
ax[1].set_ylabel('Familiarity with AI', fontsize=12)

# set entire figure's super ylabel to Odds Ratio (Log Scale)
fig.supxlabel('Interventions', fontsize=12)
fig.supylabel('Odds Ratio (Log Scale)', fontsize=12)
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3, fontsize=12, handles=[mpatches.Patch(color='C1', label='Accuracy'), mpatches.Patch(color='C2', label='Trust'), mpatches.Patch(color='C3', label='Adherence')])

save_fig("strict_misinfo_true_interventions")